# Le modèle que l'on ajuste

Le modèle est un SEIRH par âge. H réunit l'hôpital et les soins intensifs. Il ne voit que les séries publiées. Il ne lit ni le simulateur, ni les paramètres secrets.

Il est plus simple que la ville, exprès.

| | Ville (`simulator`) | Modèle (`model`) |
|---|---|---|
| Infectieux | asymptomatique, présymptomatique, symptomatique | asymptomatique et symptomatique |
| Lieux | maison, école, travail, communauté | une matrice, somme pondérée des quatre |
| Hasard de la dynamique | chaîne binomiale, et plus aux niveaux 2 et 3 | variantes A et B : pas de chaîne. Le hasard est dans l'observation |
| Âge | cinq tranches | le même découpage, mais aucune série n'est publiée par âge |

La séparation par âge repose donc sur les priors et sur la matrice de contact, pas sur une observation directe.

Les pointillés des schémas ci-dessous ne sont pas une transition de la maladie. Ce sont le test, la sensibilité et le faux négatif.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Le notebook est dans notebooks/. On remonte à la racine du dépôt.
RACINE = Path.cwd().resolve()
if not (RACINE / "pyproject.toml").exists():
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE / "src"))

%matplotlib inline
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)


In [2]:
from epidemic.viz.model_diagram import INFERENCE_EDGES, SIMULATOR_EDGES

def tableau_flux(aretes: tuple) -> pd.DataFrame:
    """Met les flèches du schéma dans un tableau."""
    return pd.DataFrame(list(aretes), columns=["de", "vers", "etiquette", "style"])

print("Ville")
print(tableau_flux(SIMULATOR_EDGES).to_string(index=False))
print()
print("Modèle d'inférence")
print(tableau_flux(INFERENCE_EDGES).to_string(index=False))


Ville
  de        vers        etiquette  style
   S           E                β  solid
   E          Ia        1 − p_sym  solid
   E          Ip            p_sym  solid
  Ip          Is présymptomatique  solid
  Ia           R                γ  solid
  Is           H           p_hosp  solid
  Is           R       1 − p_hosp  solid
   H         ICU            p_icu  solid
   H           R         guérison  solid
   H           D    décès service  solid
 ICU           R         guérison  solid
 ICU           D         décès SI  solid
  Is        Test           p_test dashed
Test     Positif      sensibilité dashed
Test FauxNegatif  1 − sensibilité dashed

Modèle d'inférence
  de        vers       etiquette  style
   S           E               β  solid
   E          Ia   σ (1 − p_sym)  solid
   E          Is         σ p_sym  solid
  Ia           R               γ  solid
  Is           H        γ p_hosp  solid
  Is           R  γ (1 − p_hosp)  solid
   H         ICU       γ_h p_icu  soli

La différence tient en une ligne : la ville a `E → présymptomatique → symptomatique`. Le modèle envoie une part des exposés directement vers les symptomatiques (`σ p_sym`) et le reste vers les asymptomatiques. Hôpital, soins intensifs, guérison et décès ont la même logique des deux côtés. L'hôpital ne transmet pas.


## L'entrée du modèle : cinq séries, pas le CSV brut

`observations_from_frame` garde les cas, les tests, les hospitalisés et les soins intensifs. Les décès cumulés deviennent le flux du jour : décès du jour = cumul du jour − cumul de la veille, sans valeur négative.


In [3]:
from epidemic.series import observations_from_frame

cours = pd.read_csv(RACINE / "data" / "external" / "data.csv")
serie = observations_from_frame(cours)
print(f"{serie.n_days} jours")
cumul = cours["num_fatalities"].to_numpy()
# Premier jour où le cumul augmente. iloc, comme le tableau numpy, exclut la borne de droite.
saut = int((cumul[1:] > cumul[:-1]).nonzero()[0][0]) + 1
fenetre = slice(saut - 1, saut + 3)
print(cours.iloc[fenetre][["Day", "num_fatalities"]].to_string(index=False))
print("Décès du jour vus par le modèle :", serie.death_increments[fenetre].round(0).tolist())


71 jours
 Day  num_fatalities
  26               0
  27               1
  28               1
  29               1
Décès du jour vus par le modèle : [0.0, 1.0, 0.0, 0.0]


## Pourquoi tant d'infections pour si peu de cas

Sur une ville de niveau 1, on compare le total caché et le total publié. Le rapport n'est pas un paramètre du fichier : il sort du dépistage. Seuls les nouveaux symptomatiques sont testés, au moins la moitié d'entre eux. Un infecté testé est confirmé avec la sensibilité du test. Les asymptomatiques et les présymptomatiques ne sont pas testés. Une admission à l'hôpital, elle, est comptée positive.

La spécificité vaut 100 % dans les priors. Un non-infecté testé n'est donc pas déclaré positif : les faux positifs de la cellule suivante doivent être nuls.


In [4]:
from epidemic.simulator import RealitySimulator

observations, cache = RealitySimulator.from_configs(seed=1, level=1).run(30)
infections = int(cache["new_infections"].sum())
confirmes = int(observations["num_positive"].sum())
print(f"Nouvelles infections     {infections:6d}")
print(f"Cas confirmés            {confirmes:6d}")
print(f"Vrais positifs           {int(cache['true_positives'].sum()):6d}")
print(f"Faux négatifs            {int(cache['false_negatives'].sum()):6d}")
print(f"Faux positifs            {int(cache['false_positives'].sum()):6d}")
if confirmes:
    print(f"Infections / cas confirmés  {infections / confirmes:.1f}")


Nouvelles infections        330
Cas confirmés                99
Vrais positifs               98
Faux négatifs                39
Faux positifs                 0
Infections / cas confirmés  3.3


## Les priors : une loi, pas une borne dure

Les consignes donnent des intervalles (« incubation de 1 à 5 jours »). Le modèle en fait des lois. L'incubation est une log-normale dont l'intervalle central à 90 % tombe sur 1–5 jours. La fraction symptomatique et la sensibilité sont des lois Beta de même moyenne que l'intervalle annoncé.

Le tableau suivant tire un intervalle à 95 % dans le code (`prior_rows`). Il est plus large que l'intervalle à 90 % calé sur les consignes : une log-normale d'incubation « 1 à 5 jours » à 90 % déborde un peu à 95 %. Le premier bloc garde les priors dont la source est exactement `consigne`. La pente d'âge est marquée `consigne du projet` dans le catalogue : le cours ne donne pas son chiffre, elle passe donc avec les hypothèses. Le détail est dans `docs/hypotheses.md`.


In [5]:
from epidemic.reporting.priors_table import prior_rows

tableau = pd.DataFrame(prior_rows(seed=0, sample_size=4_000))
# « consigne » tout court : l'intervalle vient du cours. « consigne du projet » est un choix.
consignes = tableau["source"] == "consigne"
print("Intervalle donné par les consignes")
print(
    tableau.loc[consignes, ["label", "family", "parameters", "interval"]].to_string(
        index=False
    )
)
print()
print("Hypothèses du projet, y compris la pente d'âge")
print(
    tableau.loc[~consignes, ["label", "family", "interval", "justification"]].to_string(
        index=False
    )
)


Intervalle donné par les consignes
                 label         family                        parameters             interval
    Durée d'incubation    Log-normale                μ = 0.80, σ = 0.49 2,24 (0,86 à 5,83 j)
   Durée symptomatique    Log-normale                μ = 1.84, σ = 0.28 6,32 (3,66 à 10,9 j)
Fraction symptomatique           Beta                α = 54.7, β = 36.5   0,60 (0,50 à 0,70)
 Sensibilité des tests           Beta                α = 91.5, β = 26.6   0,78 (0,70 à 0,85)
     Taux de dépistage   Beta décalée          0,50 + 0,50 × Beta(2, 2)   0,74 (0,55 à 0,96)
 Spécificité des tests Fixée, ou Beta 1, ou Beta(199, 1) si on l'estime            1 (fixée)

Hypothèses du projet, y compris la pente d'âge
                           label                     family             interval                                                             justification
                 Transmission β₀                Log-normale   0,03 (0,01 à 0,10)           Par contact pondéré.

## Trois variantes, deux façons d'ajuster

La variante A laisse bouger β(t), par une marche aléatoire hebdomadaire. La dispersion de l'observation est fixée. La variante B estime en plus la surdispersion de chaque série. La variante C ajoute un bruit sur les nouvelles infections, proche d'une chaîne binomiale. On ne lance pas un filtre particulaire tant que A et B n'ont pas montré leurs limites.

L'ajustement de référence est NUTS (4 chaînes, 500 pas d'échauffement, 500 tirages). On le juge acceptable si R̂ reste sous 1,01 et si l'ESS vaut au moins 400. Le SVI est l'approximation rapide, pour l'application. Il ne sert pas à comparer des modèles par LOO ou WAIC : il ne garantit pas la log-vraisemblance point par point.

Ce notebook ne lance pas NUTS. Un ajustement complet se commande avec :

```bash
uv run epidemic fit
```

Un essai court, en SVI, sur 21 jours :

```bash
uv run epidemic fit --method svi --days 21 --no-strict
```


In [6]:
from epidemic.config import load_inference_config
from epidemic.inference import fit_observations

reglages = load_inference_config()
print(f"Variante {reglages.variant}, β(t) = {reglages.beta_mode}, méthode = {reglages.method}")
print(
    f"NUTS : {reglages.nuts.num_chains} chaînes, "
    f"{reglages.nuts.num_warmup} pas d'échauffement, "
    f"{reglages.nuts.num_samples} tirages"
)
print(f"SVI : {reglages.svi.num_steps} pas, puis {reglages.svi.num_samples} tirages")
print(f"R̂ < {reglages.diagnostics.max_rhat}, ESS ≥ {reglages.diagnostics.min_ess}")
print()
print(fit_observations.__doc__)


Variante A, β(t) = weekly_rw, méthode = nuts
NUTS : 4 chaînes, 500 pas d'échauffement, 500 tirages
SVI : 2000 pas, puis 400 tirages
R̂ < 1.01, ESS ≥ 400.0

Ajuste le SEIRH étendu sur les séries publiées. N'ouvre aucun fichier secret.


Un paramètre est lu comme identifié quand son intervalle a posteriori fait moins de la moitié de l'intervalle a priori. Ce seuil (0,50, et 0,85 pour une identification partielle) est une convention de lecture, dans `configs/inference.yaml`. Il n'est pas une propriété de la maladie.

La sortie d'un ajustement est une loi, pas un jeu de paramètres uniques. Le notebook suivant s'en sert pour comparer des mesures.
